In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [2]:
def run_feature_engineering():
    print("Starting Label & Feature Engineering...")
    import pandas as pd
    from sklearn.preprocessing import StandardScaler
    
    df = pd.read_csv('master_data.csv')
    
    # 1. Create the composite target label 'high_friction'
    cond1 = df['onboarding_days'] > 15
    cond2 = df['training_completion_percent'] < 85
    cond3 = (df['total_tickets'] >= 2) | (df['high_priority_tickets'] >= 1)
    cond4 = df['onboarding_status'].isin(['In Progress', 'Delayed'])
    
    df['high_friction'] = (cond1 | cond2 | cond3 | cond4).astype(int)
    
    # 2. Select predictive features (Early Signals)
    columns_to_drop = [
        'onboarding_days', 
        'training_completion_percent', 
        'total_tickets', 
        'high_priority_tickets',
        'avg_resolution_hours',
        'onboarding_status',
        'onboarding_completion_date'
    ]
    features_df = df.drop(columns=columns_to_drop)
    
    # 2.5 New Feature Engineering (Early Signals Interactions)
    features_df['early_active_minutes_per_login'] = features_df['early_active_minutes'] / (features_df['early_login_count'] + 1e-5)
    features_df['early_tool_actions_per_unique_tool'] = features_df['early_tool_actions'] / (features_df['early_unique_tools'] + 1e-5)
    features_df['zero_early_activity'] = (features_df['early_tool_actions'] == 0).astype(int)
    features_df['early_activity_x_joblevel'] = features_df['early_active_minutes'] * features_df['JobLevel']
    
    # 3. Handle Categorical Encoding with One-Hot (better for linear models/tree combinations)
    categorical_cols = ['Department', 'JobRole', 'Gender', 'BusinessTravel', 'orientation_completed', 'manager_assigned', 'buddy_assigned', 'first_week_checkin']
    features_df = pd.get_dummies(features_df, columns=categorical_cols, drop_first=True)
    
    # 4. Feature scaling for distance-based/regularized models
    cols_to_scale = [c for c in features_df.columns if c not in ['employee_id', 'high_friction']]
    scaler = StandardScaler()
    features_df[cols_to_scale] = scaler.fit_transform(features_df[cols_to_scale])
    
    print(f"\nFinal Features Dataset Shape: {features_df.shape}")
    features_df.to_csv('engineered_data.csv', index=False)
    print("\nSaved 'engineered_data.csv'.")


In [3]:
if __name__ == '__main__':
    run_feature_engineering()

Starting Label & Feature Engineering...

Final Features Dataset Shape: (1470, 28)



Saved 'engineered_data.csv'.
